In [1]:
pip install pandas yfinance datasets spacy tqdm opendatasets transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python -m spacy download en_core_web_md

c:\Users\Calmen\AppData\Local\Programs\Python\Python311\python.exe: No module named spacy


MasterData Building

In [2]:
pip install dotenv

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import requests
import pandas as pd
import time
from datetime import datetime
from calendar import monthrange
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
BASE_URL = "https://www.alphavantage.co/query"

MY_TICKERS = {
    'CL=F', 'USO', 'XOM', 'CVX', 'BP', 'SHEL', 'COP', 'EOG', 'OXY', 'XLE', 'XOP',
    'HO=F', 'CRAK', 'VLO', 'MPC', 'PSX', 'NG=F', 'UNG', 'LNG', 'TELL', 'EQT', 'RRC'
}

def harvest_chunk(year: int, month: int) -> list:
    """Fetch up to 1000 articles for one month."""
    last_day = monthrange(year, month)[1]
    time_from = f"{year}{month:02d}01T0000"
    time_to = f"{year}{month:02d}{last_day:02d}T2359"

    params = {
        "function": "NEWS_SENTIMENT",
        "topics": "energy",
        "time_from": time_from,
        "time_to": time_to,
        "limit": 1000,
        "sort": "LATEST",
        "apikey": API_KEY,
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        # Alpha Vantage often returns a "Note" when rate limited
        if "Note" in data:
            print(f"Rate limit / API note for {year}-{month:02d}: {data['Note']}")
            return []

        if "Information" in data:
            print(f"API info for {year}-{month:02d}: {data['Information']}")
            return []

        return data.get("feed", [])

    except Exception as e:
        print(f"Error on {year}-{month:02d}: {e}")
        return []

def check_relevance(ticker_list) -> bool:
    if not isinstance(ticker_list, list):
        return False
    found = {item.get("ticker") for item in ticker_list if isinstance(item, dict)}
    return not found.isdisjoint(MY_TICKERS)

if not API_KEY:
    raise ValueError("Missing ALPHAVANTAGE_API_KEY environment variable")

current_year = datetime.now().year
current_month = datetime.now().month

# Example: harvest 2024 through current year
years_to_harvest = [2023,2025]

all_data = []

print(f"Starting harvest for years: {years_to_harvest}")

for year in years_to_harvest:
    for month in range(1, 13):
        if year == current_year and month > current_month:
            continue

        print(f"Pulling {year}-{month:02d}...")
        batch = harvest_chunk(year, month)
        all_data.extend(batch)

        print(f"Collected {len(batch)} articles. Waiting 15s...")
        time.sleep(15)

df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    print("No data found. Check API key, rate limits, or query filters.")
else:
    if "ticker_sentiment" not in df_raw.columns:
        raise KeyError("ticker_sentiment column missing from API response")

    print("Filtering for relevant tickers...")
    df_raw["is_relevant"] = df_raw["ticker_sentiment"].apply(check_relevance)
    master_df = df_raw[df_raw["is_relevant"]].copy()

    if "url" in master_df.columns:
        master_df = master_df.drop_duplicates(subset=["url"])

    if "time_published" in master_df.columns:
        master_df["dt"] = pd.to_datetime(
            master_df["time_published"],
            format="%Y%m%dT%H%M%S",
            errors="coerce"
        )
        master_df = master_df.sort_values("dt", ascending=False)

    filename = f"Energy_BigData_2023_2025.csv"
    master_df.to_csv(filename, index=False)

    print("\nHARVEST COMPLETE")
    print(f"Total raw articles: {len(df_raw)}")
    print(f"Total relevant articles: {len(master_df)}")
    print(f"Saved to: {filename}")

Starting harvest for years: [2023, 2025]
Pulling 2023-01...


KeyboardInterrupt: 

In [1]:
import pandas as pd
df=pd.read_csv("Energy_BigData_2023_2025.csv")

In [2]:
df.head()

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,is_relevant,dt
0,Bill Nygren's Portfolio - Oakmark Select Fund,https://www.heygotrade.com/en/insider/bill-nyg...,20251231T233659,['Rendy Andriyanto Gotrade Team'],This article details Bill Nygren's portfolio f...,https://files-tr8.s3.ap-southeast-1.amazonaws....,Gotrade,General,Gotrade,"[{'topic': 'financial_markets', 'relevance_sco...",0.221761,Somewhat-Bullish,"[{'ticker': 'ABNB', 'relevance_score': '0.7312...",True,2025-12-31 23:36:59
1,Here's What Wall Street Thinks About Californi...,https://finviz.com/news/263019/heres-what-wall...,20251231T231017,['Talha Qureshi'],Wall Street analysts have reiterated Buy ratin...,https://d2gr5kl7dt2z3t.cloudfront.net/blog/wp-...,Finviz,General,Finviz,"[{'topic': 'mergers_and_acquisitions', 'releva...",0.295699,Somewhat-Bullish,"[{'ticker': 'CRC', 'relevance_score': '1.00000...",True,2025-12-31 23:10:17
2,NextEra Energy announces $4 billion at-the-mar...,https://www.investing.com/news/sec-filings/nex...,20251231T220809,['Investing.com'],NextEra Energy (NYSE:NEE) announced a $4 billi...,https://i-invdn-com.investing.com/news/World_N...,Investing.com,General,Investing.com,"[{'topic': 'earnings', 'relevance_score': '0.8...",0.411043,Bullish,"[{'ticker': 'NEE', 'relevance_score': '1.00000...",True,2025-12-31 22:08:09
3,Unusual Activity in Occidental Petroleum Call ...,https://www.barchart.com/story/news/36844081/u...,20251231T183218,"['Mark R. Hake', 'CFA']",Investors are showing unusual activity in Occi...,https://media.barchart.com/contributors-admin/...,Barchart.com,General,Barchart.com,"[{'topic': 'financial_markets', 'relevance_sco...",0.496636,Bullish,"[{'ticker': 'OXY', 'relevance_score': '1.00000...",True,2025-12-31 18:32:18
4,"Seasonal Demand Swings, Increasing LNG Supplie...",https://www.naturalgasintel.com/news/seasonal-...,20251231T182900,['Jamison Cocklin'],U.S. LNG feed gas demand surged by nearly 5 Bc...,https://apppack-app-naturalgasintel-prod-publi...,Natural Gas Intelligence,General,Natural Gas Intelligence,"[{'topic': 'energy_transportation', 'relevance...",0.097159,Neutral,"[{'ticker': 'LNG', 'relevance_score': '0.32175...",True,2025-12-31 18:29:00


In [3]:
import ast
import pandas as pd

SC_MAP = {
    # Upstream
    "CL=F": ("crude oil", "Upstream"),
    "USO": ("crude oil", "Upstream"),
    "XOM": ("crude oil", "Upstream"),
    "CVX": ("crude oil", "Upstream"),
    "COP": ("crude oil", "Upstream"),
    "EOG": ("crude oil", "Upstream"),
    "OXY": ("crude oil", "Upstream"),
    "BP": ("crude oil", "Integrated"),
    "SHEL": ("crude oil", "Integrated"),

    # LNG / natural gas
    "EQT": ("LNG", "Upstream"),
    "RRC": ("LNG", "Upstream"),
    "LNG": ("LNG", "Midstream"),
    "TELL": ("LNG", "Midstream"),
    "UNG": ("LNG", "Midstream"),
    "NG=F": ("LNG", "Midstream"),

    # Downstream / refined products
    "VLO": ("diesel", "Downstream"),
    "MPC": ("diesel", "Downstream"),
    "PSX": ("diesel", "Downstream"),
    "HO=F": ("diesel", "Downstream"),
    "CRAK": ("diesel", "Downstream"),
}


def safe_parse(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []


def extract_commodities(ticker_sentiment, sc_map=SC_MAP):
    ticker_list = safe_parse(ticker_sentiment)
    commodities = []

    for item in ticker_list:
        if isinstance(item, dict):
            ticker = item.get("ticker")
            if ticker in sc_map:
                commodity = sc_map[ticker][0]
                if commodity not in commodities:
                    commodities.append(commodity)

    return ",".join(commodities) if commodities else None


def parse_energy_to_merged_format(df_energy):
    df = df_energy.copy()

    df["commodity"] = df["ticker_sentiment"].apply(extract_commodities)
    df["date"] = pd.to_datetime(df["dt"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["description"] = df["summary"]

    result = df[["date", "title", "description", "commodity"]].copy()

    result = result.dropna(subset=["date", "title", "description"])
    result = result.reset_index(drop=True)

    return result
df_parsed = parse_energy_to_merged_format(df)

In [4]:
df_parsed.head()

,date,title,description,commodity
0,2025-12-31,Bill Nygren's Portfolio - Oakmark Select Fund,This article details Bill Nygren's portfolio f...,diesel
1,2025-12-31,Here's What Wall Street Thinks About Californi...,Wall Street analysts have reiterated Buy ratin...,crude oil
2,2025-12-31,NextEra Energy announces $4 billion at-the-mar...,NextEra Energy (NYSE:NEE) announced a $4 billi...,crude oil
3,2025-12-31,Unusual Activity in Occidental Petroleum Call ...,Investors are showing unusual activity in Occi...,crude oil
4,2025-12-31,"Seasonal Demand Swings, Increasing LNG Supplie...",U.S. LNG feed gas demand surged by nearly 5 Bc...,"LNG,crude oil"


In [5]:
import pandas as pd

# Load original merged_df if needed
merged_df = pd.read_csv("merged_df.csv")
merged_df["date"] = pd.to_datetime(
    merged_df["date"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_parsed["date"] = pd.to_datetime(
    df_parsed["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

final_df = pd.concat(
    [
        merged_df[["date", "title", "description", "commodity"]],
        df_parsed[["date", "title", "description", "commodity"]],
    ],
    ignore_index=True
).drop_duplicates(subset=["date", "title", "description", "commodity"])

final_df = final_df.sort_values("date").reset_index(drop=True)

print(merged_df["date"].min(), merged_df["date"].max())
print(final_df["date"].min(), final_df["date"].max())
print(final_df["date"].dt.year.value_counts().sort_index())



2022-01-03 00:00:00 2026-03-31 00:00:00
2022-01-03 00:00:00 2026-03-31 00:00:00
date
2022     83
2023    222
2024    225
2025    141
2026     57
Name: count, dtype: int64


In [8]:
final_df.shape[0]

728

In [9]:
final_df=final_df[final_df["commodity"].notna() & (final_df["commodity"] != "")]

In [10]:
final_df['date'] = pd.to_datetime(final_df['date'], errors='coerce')
print(final_df['date'].min(), final_df['date'].max())
final_df['year'] = final_df['date'].dt.year
final_df['month'] = final_df['date'].dt.month
final_df['year_month'] = final_df['date'].dt.to_period('M').astype(str)
print(sorted(final_df['year_month'].dropna().unique()))

2022-01-03 00:00:00 2026-03-31 00:00:00
['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-03']


Labelling

Using LLMs to label domain and extract risk keywords

In [11]:
import json
import time
import pandas as pd

SYSTEM_INSTRUCTION_EXPAND = """
You are an expert Energy Market & Supply Chain Analyst. 
Your task is to analyze news title and description to categorize energy news for a high-precision Risk Management RAG system.

Assign each item:
1. news_category
2. risk_category
3. risk_severity
4. market_sentiment

--- PREDEFINED NEWS CATEGORIES ---
1. Geopolitics & policy
2. Supply (production / upstream)
3. Refining (downstream)
4. Inventory & storage
5. Demand / macro activity
6. LNG & natural gas
7. Weather
8. Shipping & logistics
9. Financial flows / positioning
10. Currency & interest rates
11. Accidents / disruptions
12. Energy transition / structural

--- PREDEFINED RISK CATEGORIES ---
- Supply Chain Blockage
- Production Shortfall
- Refining Outage
- Geopolitical Conflict
- Macro-Economic Cooling
- Inventory Shock
- Infrastructure Damage
- Weather Extremity
- Regulatory Constraint
- No Significant Risk

--- PREDEFINED RISK SEVERITY ---
- Low
- Medium
- High

--- PREDEFINED MARKET SENTIMENT ---
- Bullish
- Bearish
- Neutral

Rules:
- Choose exactly one news_category from the predefined list.
- Choose exactly one risk_category from the predefined list.
- Choose exactly one risk_severity from: Low, Medium, High.
- Choose exactly one market_sentiment from: Bullish, Bearish, Neutral.
- Base the label on the title + description only.
- If the article is mainly about company earnings, stock movement, analyst ratings, or financing, prefer:
  news_category = Financial flows / positioning
- If the article is mainly about LNG, gas supply, pipelines, or gas exports/imports, prefer:
  news_category = LNG & natural gas
- If no material risk is present, use:
  risk_category = No Significant Risk
  risk_severity = Low
- market_sentiment should reflect likely commodity-market impact, not just company-level stock tone.

--- OUTPUT FORMAT ---
Return a JSON list.
Each object must contain exactly:
{
  "id": <input id>,
  "news_category": "<one predefined news category>",
  "risk_category": "<one predefined risk category>",
  "risk_severity": "<Low|Medium|High>",
  "market_sentiment": "<Bullish|Bearish|Neutral>"
}
"""


In [12]:
def build_batch_prompt(batch_df):
    prompt_input = ""
    for idx, row in batch_df.iterrows():
        title = str(row.get("title", "")).strip()
        description = str(row.get("description", "")).strip()
        commodities = str(row.get("relevant_commodities", "")).strip()

        prompt_input += (
            f"ID: {idx} | "
            f"Title: {title} | "
            f"Description: {description} | "
            f"Relevant Commodities: {commodities}\n"
        )

    return SYSTEM_INSTRUCTION_EXPAND + "\n\nAnalyze these items:\n" + prompt_input


In [13]:
def expand_categories_with_llm(
    df,
    client,
    model="gpt-4o-mini",
    batch_size=20,
    sleep_seconds=0.5
):
    working_df = df.copy()

    if "commodity" in working_df.columns:
        working_df = working_df.rename(columns={"commodity": "relevant_commodities"})

    required_cols = ["date", "title", "description", "relevant_commodities"]
    missing = [c for c in required_cols if c not in working_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    all_results = []

    for start in range(0, len(working_df), batch_size):
        batch = working_df.iloc[start:start + batch_size]
        prompt = build_batch_prompt(batch)

        response = client.responses.create(
            model=model,
            input=prompt,
        )

        text = response.output_text.strip()
        text = text.replace("```json", "").replace("```", "").strip()

        try:
            batch_results = json.loads(text)
        except json.JSONDecodeError as e:
            print(f"JSON parse error for batch starting at row {start}: {e}")
            print(text[:1000])
            continue

        all_results.extend(batch_results)
        print(f"Processed rows {start} to {start + len(batch) - 1}")
        time.sleep(sleep_seconds)

    labels_df = pd.DataFrame(all_results)

    if labels_df.empty:
        raise ValueError("No labels returned from LLM.")

    if "id" not in labels_df.columns:
        raise ValueError("LLM output must contain 'id'.")

    labels_df["id"] = labels_df["id"].astype(int)

    merged = (
        working_df.reset_index()
        .merge(labels_df, left_on="index", right_on="id", how="left")
        .drop(columns=["index", "id"])
    )

    final_cols = [
        "date",
        "title",
        "description",
        "relevant_commodities",
        "news_category",
        "risk_category",
        "risk_severity",
        "market_sentiment",
    ]

    return merged[final_cols]


In [ ]:
# pip install openai

     ---------------------------------------- 1.1/1.1 MB 14.6 MB/s eta 0:00:00
     ---------------------------------------- 204.6/204.6 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
import requests
import pandas as pd
import time
from datetime import datetime
from calendar import monthrange
from dotenv import load_dotenv
import os

load_dotenv()
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

expanded_df = expand_categories_with_llm(
    df=final_df,
    client=client,
    model="gpt-4o-mini",
    batch_size=20
)

expanded_df.head()


Processed rows 0 to 19
Processed rows 20 to 39
Processed rows 40 to 59
Processed rows 60 to 79
Processed rows 80 to 99
Processed rows 100 to 119
Processed rows 120 to 139
Processed rows 140 to 159
Processed rows 160 to 179
Processed rows 180 to 199
Processed rows 200 to 219
Processed rows 220 to 239
Processed rows 240 to 259
Processed rows 260 to 279
Processed rows 280 to 299
JSON parse error for batch starting at row 300: Expecting value: line 1 column 1 (char 0)
Here’s the analysis of the provided news items categorized accordingly:


[
    {
        "id": 300,
        "news_category": "Financial flows / positioning",
        "risk_category": "No Significant Risk",
        "risk_severity": "Low",
        "market_sentiment": "Bearish"
    },
    {
        "id": 301,
        "news_category": "Supply (production / upstream)",
        "risk_category": "No Significant Risk",
        "risk_severity": "Low",
        "market_sentiment": "Bullish"
    },
    {
        "id": 302,
        "news

,date,title,description,relevant_commodities,news_category,risk_category,risk_severity,market_sentiment
0,2022-01-03,"After Maharashtra, distributors might boycott ...",FMCG product distributors are planning to boyc...,crude oil,Financial flows / positioning,No Significant Risk,Low,Neutral
1,2022-01-03,Colgate items to go off shelves in Maharashtra,FMCG distributors in Maharashtra have announce...,crude oil,Financial flows / positioning,No Significant Risk,Low,Neutral
2,2022-01-04,Colgate in talks with India sales agents after...,Colgate-Palmolive India is in talks with its s...,crude oil,Financial flows / positioning,No Significant Risk,Low,Neutral
3,2022-01-04,Colgate in talks with India sales agents after...,Colgate-Palmolive India is in discussions with...,crude oil,Financial flows / positioning,No Significant Risk,Low,Neutral
4,2022-01-04,US becomes world’s top LNG exporter for first ...,The United States became the world's top expor...,LNG,LNG & natural gas,No Significant Risk,Low,Bullish


In [17]:
def show_counts_only(df):
    cols = ["news_category", "risk_category", "risk_severity", "market_sentiment"]

    for col in cols:
        print(f"\n=== {col} ===")
        print(df[col].fillna("Missing").value_counts())

    print("\n=== relevant_commodities ===")
    commodity_counts = (
        df["relevant_commodities"]
        .fillna("Missing")
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
        .value_counts()
    )
    print(commodity_counts)
show_counts_only(expanded_df)





=== news_category ===
news_category
Financial flows / positioning     241
LNG & natural gas                 128
Geopolitics & policy              118
Supply (production / upstream)    102
Energy transition / structural     47
Refining (downstream)              25
Missing                            20
Demand / macro activity            16
No Significant Risk                 9
Accidents / disruptions             7
Shipping & logistics                6
Inventory & storage                 4
Regulatory Constraint               2
Weather                             2
Technology & Innovation             1
Name: count, dtype: int64

=== risk_category ===
risk_category
No Significant Risk        526
Geopolitical Conflict       46
Regulatory Constraint       42
Production Shortfall        37
Supply Chain Blockage       22
Missing                     20
Infrastructure Damage       13
Macro-Economic Cooling      11
Refining Outage              6
Inventory Shock              3
Demand / macro activ

In [18]:
expanded_df.to_csv("final_expanded_df.csv", index=False)